# Fraud model — Phase 2 temporal training

**Status: V1 was evaluated and not promoted; this notebook now contains the V2 customer-history candidate.** It compares the required baselines, logistic-regression candidates, decision-tree candidates, and one bounded Random Forest challenger on the fixed temporal split. Preprocessing is fitted on train only. Hyperparameters and model choice use validation only. The final-test period is excluded from the source query and must remain untouched until Phase 3.

## Questions this experiment answers

1. Does either required model improve over constant/always-negative baselines on temporal validation?
2. At 0.1%, 0.5%, 1%, 2%, 5%, 10%, and 20% review capacity, what precision/recall is achieved, and how sensitive are results to tied scores?
3. Do results vary materially by country or channel, with denominators shown?

## Method and safeguards

- Run on Databricks Serverless environment version 4, which supports PySpark ML and MLflow for Spark.
- Read one pinned Delta snapshot (version 1); use a single transactions table, so joins cannot multiply rows.
- Train: before 2025-07-01; validation: 2025-07-01 through 2025-12-31; no final-test data.
- V2 features use strictly earlier same-customer transaction counts (1h/24h/7d), elapsed time since the previous transaction, and 30-day amount history within the same currency. Same-timestamp transactions are excluded.
- Fit median imputers and categorical vocabularies on train only; one-hot encode categories and preserve unknowns.
- Compare B0/B1, two M1 configurations (unweighted/balanced at C=1), and two conservative M2 configurations (unweighted/balanced, depth 5, leaf size 1,000). This small candidate set is bounded because Serverless does not support explicit DataFrame persistence; expand only if results and compute justify it.
- Primary ranking metric is Spark `areaUnderPR`; the highest-areaUnderPR candidate is selected. Precision/recall and best/worst tie-bound estimates are reported at several review capacities. Results are offline synthetic-data evidence, not calibrated probabilities or a production fraud decision.
- Log attempt parameters, metrics, aggregate evidence, and the selected model’s coefficient/tree specification as JSON in MLflow under `/Shared/fraud-eda/phase2-training`. The binary Spark model is not logged because this principal has no writable UC Volume for MLflow staging.
- The model specification is not an executable binary and is not registered; creating a loadable artifact requires an authorized MLflow staging Volume. No database table is written.


In [ ]:
%run ./train_phase2

In [ ]:
import json
import mlflow
result = run_training(spark, mlflow)
print(json.dumps(result, indent=2, allow_nan=False))
dbutils.notebook.exit(json.dumps(result, allow_nan=False))